# T2.2 Evaluation Notebook

Run MRR@5 and Recall@5 on `data/gold_matches.csv`, then inspect the 3 confusion cases.

To reproduce:
```bash
pip install -r requirements.txt
python src/generate_data.py
```
then run this notebook top-to-bottom.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from evaluate import evaluate, print_report
results = evaluate(k=5)
print_report(results)

## Summary table

In [ ]:
import pandas as pd

summary = pd.DataFrame([{
    'Metric': 'MRR@5', 'Value': f"{results['mrr_at_k']:.3f}"
}, {
    'Metric': 'Recall@5', 'Value': f"{results['recall_at_k']:.3f}"
}, {
    'Metric': 'Tenders', 'Value': results['n_tenders']
}, {
    'Metric': 'Profiles', 'Value': results['n_profiles']
}, {
    'Metric': 'End-to-end (ms)', 'Value': results['total_end_to_end_ms']
}])
summary

## Per-profile breakdown

In [ ]:
df = pd.DataFrame(results['per_profile'])
df[['profile_id', 'sector', 'country', 'primary_lang', 'rr', 'recall', 'predicted', 'gold', 'hits']]

## Three confusion cases

Profiles where the ranker most disagreed with the expert labels. These are the cases to
discuss in the Live Defence — each gets a short diagnosis below the table.

In [ ]:
cdf = pd.DataFrame(results['confusion_cases'])
cdf[['profile_id', 'sector', 'primary_lang', 'rr', 'recall', 'predicted', 'gold', 'hits']]

### Diagnosis

For each confusion case, the miss is not sector confusion (all predictions are in-sector)
but **ranking refinement**: two or more tenders share sector + budget band and the ranker
can't cleanly separate them from lexical overlap alone.

Single feature that would demote the worst miss: **past-funding-to-budget-target ratio** as
a hard gate above 10x — currently it decays smoothly, which lets big-ticket tenders float
into the top-5 for early-stage profiles.